# 04 · Safety & Hallucination Mitigation

> **Source notes:** `SafetyAndHallucination.md`

A system that is right 95% of the time is dangerous if users can't tell which 5% is wrong. This notebook demonstrates the layered mitigation stack:

- **Hallucination taxonomy** — generating examples of each hallucination type
- **Grounding constraints** — prompt-level mitigation
- **NLI-based claim verification** — post-generation check using a cross-encoder
- **Self-consistency sampling** — flag low-agreement answers
- **Confidence elicitation** — ask the model to rate its own certainty

All LLM calls use **Ollama** (local, no API key needed).

## 0 · Environment Setup

```bash
ollama pull phi3:mini
ollama serve
```

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "ollama", "sentence-transformers", "-q"],
    check=True
)
print("Packages ready.")

import ollama, json, re, textwrap
from collections import Counter

MODEL = "phi3:mini"

def chat(system: str, user: str, temperature: float = 0.0, max_tokens: int = 200) -> str:
    resp = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        options={"temperature": temperature, "num_predict": max_tokens}
    )
    return resp["message"]["content"].strip()

print("Helper ready.")

## 1 · Hallucination Taxonomy — Generating Examples

| Type | Description |
|---|---|
| **Factual hallucination** | Plausible-sounding but wrong fact |
| **Confabulation** | Fabricated citation with correct-looking format |
| **Sycophantic hallucination** | Agreeing with a false premise |
| **Specification overreach** | Adding unrequested content beyond constraints |

We provoke each type to understand what we're defending against.

In [ ]:
# 1a — Factual hallucination: ask for a precise fact in a low-data domain
print("=== 1a. Factual hallucination (specific date provocation) ===")
ans = chat("", "What exact date did Mamma Rosa's Pizzeria open their Vancouver location?", max_tokens=60)
print(f"  {ans}")
print("  ⚠ Model has no data on this — any date given is hallucinated.\n")

# 1b — Sycophantic hallucination: assert a false fact and see if the model confirms it
print("=== 1b. Sycophantic hallucination (false-premise agreement) ===")
ans = chat(
    "",
    "I read that the Eiffel Tower is 550 metres tall. Is that right?",
    temperature=0.0,
    max_tokens=80
)
print(f"  {ans}")
print("  (Actual height is ~330 m. Did the model correct the false premise?)\n")

# 1c — Specification overreach: ask for exactly 2 items
print("=== 1c. Specification overreach (constraint violation) ===")
ans = chat(
    "Respond with EXACTLY 2 bullet points. No more.",
    "List the benefits of eating pizza.",
    temperature=0.7,
    max_tokens=120
)
bullets = [l for l in ans.split("\n") if l.strip().startswith(("•", "-", "*", "1", "2", "3"))]
print(f"  Response has {len(bullets)} bullet(s) (expected 2).")
print(f"  {ans[:200]}")

## 2 · Prompt-Level Mitigation — Grounding Constraints

The single most effective prompt-level mitigation:

```
"Base your answer ONLY on the provided context.
 If the answer is not present, say: 'I don't have that information.'"
```

We compare an ungrounded prompt vs a grounded prompt with the same out-of-context query.

In [ ]:
MENU_CONTEXT = (
    "Menu: Margherita £9.99 (gf base +£1.50), Pepperoni £11.99, Veggie Supreme £10.49. "
    "Allergens: Pepperoni contains dairy, gluten, pork. Margherita contains dairy, gluten. "
    "Delivery: central zone only. Min order £15."
)

# Query about something NOT in the context
query = "Does the Spicy Prawn pizza contain shellfish?"

ungrounded_system = "You are a helpful pizza ordering assistant."
grounded_system   = (
    "You are a pizza ordering assistant for Mamma Rosa's.\n"
    "IMPORTANT: Base your answer ONLY on the provided context. "
    "If the answer is not in the context, respond with: 'I don't have that information.' "
    "Do not use outside knowledge or guess."
)

user_msg = f"Context:\n{MENU_CONTEXT}\n\nQuestion: {query}"

print("── Ungrounded ──────────────────────────────────────")
print(textwrap.fill(chat(ungrounded_system, user_msg, max_tokens=80), 70))
print()
print("── Grounded ────────────────────────────────────────")
print(textwrap.fill(chat(grounded_system, user_msg, max_tokens=80), 70))

## 3 · NLI-Based Claim Verification

After generation, verify each claim in the output is **entailed** by the source context using a Natural Language Inference (NLI) cross-encoder model.

Label mapping:
- **ENTAILMENT** → claim is supported by context ✓
- **NEUTRAL** → claim may or may not be true based on context ⚠
- **CONTRADICTION** → claim is contradicted by context ✗

We use `cross-encoder/nli-deberta-v3-small` (runs locally, ~150 MB).

In [ ]:
from sentence_transformers import CrossEncoder

# Load NLI model (downloads once, ~150 MB)
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small")
NLI_LABELS = ["CONTRADICTION", "ENTAILMENT", "NEUTRAL"]

def verify_claim(claim: str, context: str) -> dict:
    """Return NLI label and confidence score for (context, claim) pair."""
    scores = nli_model.predict([(context, claim)])[0]
    label  = NLI_LABELS[scores.argmax()]
    conf   = float(scores.max())
    return {"label": label, "confidence": round(conf, 3), "scores": dict(zip(NLI_LABELS, scores.tolist()))}

# Test claims against the menu context
claims = [
    "The Margherita pizza is available with a gluten-free base.",   # TRUE — in context
    "The Pepperoni pizza contains pork.",                           # TRUE — in context
    "Delivery is available to all zones.",                          # FALSE — central only
    "The Spicy Prawn pizza contains shellfish.",                    # NOT IN CONTEXT
    "Veggie Supreme costs £10.49.",                                 # TRUE — in context
]

print(f"{'Claim':<55} {'Label':<15} {'Conf':>5}")
print("-" * 80)
for claim in claims:
    res = verify_claim(claim, MENU_CONTEXT)
    icon = {"ENTAILMENT": "✓", "NEUTRAL": "⚠", "CONTRADICTION": "✗"}[res["label"]]
    short = claim[:52] + "..." if len(claim) > 52 else claim
    print(f"{short:<55} {icon} {res['label']:<13} {res['confidence']:>5.3f}")

## 4 · Self-Consistency Sampling — Detecting Uncertain Answers

Generate **N answers at temperature > 0**, then take the majority vote. Low-agreement answers (no clear majority) are flagged for human review.

This is useful for high-stakes queries where a single-pass answer may be wrong.

In [ ]:
def self_consistency(question: str, n: int = 5, temperature: float = 0.7) -> dict:
    """Sample N answers and compute majority-vote consistency."""
    system = (
        "Answer concisely in one sentence. "
        "For yes/no questions start with 'Yes' or 'No'."
    )
    answers = []
    for _ in range(n):
        resp = ollama.chat(
            model=MODEL,
            messages=[{"role": "system", "content": system},
                      {"role": "user",   "content": question}],
            options={"temperature": temperature, "num_predict": 40}
        )
        answers.append(resp["message"]["content"].strip())

    # Simple leading-word majority vote (Yes/No questions)
    first_words = [a.split()[0].lower().rstrip(".,") for a in answers]
    vote = Counter(first_words).most_common(1)[0]
    agreement = vote[1] / n
    return {"answers": answers, "majority": vote[0], "agreement": agreement}

# High-certainty question (should be consistent)
q1 = "Is the Eiffel Tower located in France?"
# Uncertain / tricky question
q2 = f"Context: {MENU_CONTEXT}\n\nQuestion: Can I order a Spicy Prawn pizza for delivery?"

for question in [q1, q2]:
    result = self_consistency(question)
    flag   = "⚠ LOW AGREEMENT — escalate" if result["agreement"] < 0.6 else "✓ High agreement"
    print(f"Q: {question[:80]}")
    print(f"   Majority={result['majority']!r}  Agreement={result['agreement']:.0%}  {flag}")
    print(f"   Samples: {result['answers'][:3]}")
    print()

## 5 · Confidence Elicitation

Ask the model to rate its own confidence. Low-confidence answers trigger a fallback (human escalation or "I don't know" response). Note: self-reported confidence is not perfectly calibrated, but it correlates with actual accuracy enough to be useful.

In [ ]:
conf_system = """
Answer the question, then on a new line rate your confidence as one of:
CONFIDENCE: high | medium | low

Use 'low' if the answer requires information you may not have.
Use 'medium' if you are mostly sure but uncertain on some details.
Use 'high' only if you are certain.
"""

questions = [
    "What is the capital of France?",
    "What is the exact population of Reykjavik as of March 2026?",
    f"Context: {MENU_CONTEXT}\n\nQuestion: What does the Margherita cost with a gluten-free base?",
    f"Context: {MENU_CONTEXT}\n\nQuestion: What wine pairings do you recommend with the Pepperoni?",
]

import re

def elicit_confidence(question: str) -> dict:
    raw = chat(conf_system, question, max_tokens=100)
    match = re.search(r"CONFIDENCE:\s*(high|medium|low)", raw, re.IGNORECASE)
    confidence = match.group(1).lower() if match else "unknown"
    answer_text = raw.split("CONFIDENCE:")[0].strip() if "CONFIDENCE:" in raw else raw
    return {"answer": answer_text, "confidence": confidence}

for q in questions:
    result = elicit_confidence(q)
    icon   = {"high": "✓", "medium": "⚠", "low": "✗", "unknown": "?"}[result["confidence"]]
    print(f"Q: {q.split(chr(10))[-1][:75]}")
    print(f"   {icon} Confidence: {result['confidence'].upper()}")
    print(f"   Answer: {result['answer'][:100]}")
    print()

## Summary — Layered Mitigation Stack

| Layer | Technique | This Notebook |
|---|---|---|
| Prompt-level | Grounding constraint | Section 2 |
| Post-generation | NLI claim verification | Section 3 |
| Post-generation | Self-consistency sampling | Section 4 |
| Post-generation | Confidence elicitation | Section 5 |

Apply all four layers in production for high-stakes outputs. No single technique eliminates hallucination — defence in depth is the only reliable strategy.

**Next:** [EvaluatingAISystems/notebook.ipynb](../EvaluatingAISystems/notebook.ipynb) — systematic evaluation beyond ad-hoc testing.